
# Transform Races Data

1. Read bronze races table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId -> circuit_id, raceName -> race_name)
4. Rename columns to make them more meaningful (date -> race_date)
5. Remove duplicate records
6. Transform values of columns race_name to Title Case
7. Write the transformed data to silver races table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"


### Step 1 - Read bronze races table

In [0]:
# circuits_df = spark.table(bronze_table)
races_df = spark.read.table(bronze_table).filter((col("batch_id") == v_batch_id))

### Step 2 - Keep only the columns required for analytics (Drop url column)



In [0]:
races_selected_df = races_df.select(
    "season",
    "round",
    "raceName",
    "date",
    "circuitId",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
)


### Step 3

- Standardise column names using snake_case (circuitId -> circuit_id, raceName -> race_name)
- Rename columns to make them more meaningful (date -> race_date)

In [0]:
races_renamed_df = races_selected_df\
    .withColumnsRenamed(
        {
            "circuitId":"circuit_id",
            "raceName":"race_name",
            "date":"race_date"
        }
    )


### Step 5 - Remove Duplicate Records


In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])


### Step  6 - Transform Value of column race_name to Title Case


In [0]:
races_final_df = (
    races_distinct_df
    .withColumn('race_name', initcap('race_name'))
)



### Step  7 - Write the transformed data to silver races table

In [0]:
write_to_silver(
    races_final_df,
    silver_table,
    "t.season = s.season AND t.round = s.round",
    [
        "race_name",
        "race_date",
        "circuit_id",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))